# IQ Distribution in the U.S. by Race

Data source: **APA Task Force Report (1996)** — *"Intelligence: Knowns and Unknowns"*  
Published in *American Psychologist*, February 1996.  

This visualization models normal (Gaussian) distributions using the widely-cited mean IQ estimates by racial group, with a standard deviation of 15 for all groups.

In [25]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm
import plotly.io as pio
import os

pio.renderers.default = "vscode" if os.environ.get("VSCODE_PID") else "notebook_connected"

# ── APA 1996 Data ──────────────────────────────────────────────────────
# Source: APA Task Force, "Intelligence: Knowns and Unknowns" (1996)
# Mean IQ estimates by racial group, SD = 15 for all
groups = {
    "Asian American":  {"mean": 106, "sd": 15, "color": "#4F7F84"},  # Blue Green
    "White":           {"mean": 100, "sd": 15, "color": "#494C30"},  # Olive Green
    "Hispanic":        {"mean": 89,  "sd": 15, "color": "#F2CFD0"},  # Rose Quartz
    "Black":           {"mean": 85,  "sd": 15, "color": "#C4787A"},  # Mauve
}

# ── Generate distribution curves ──────────────────────────────────────
x = np.linspace(40, 160, 600)

# ── Dark beige palette ────────────────────────────────────────────────
BG_COLOR      = "#EEE5E0"   # EEE5E0 #F1E3D0
PAPER_COLOR   = "#1E1B16"
GRID_COLOR    = "rgba(255, 235, 200, 0.08)"
TEXT_COLOR    = "#FFE8C8"   # Warm cream
TITLE_COLOR   = "#FFF5E6"
AXIS_COLOR    = "rgba(255, 235, 200, 0.25)"
SUBTITLE_CLR  = "rgba(255, 232, 200, 0.55)"

# ── Build the figure ──────────────────────────────────────────────────
fig = go.Figure()

for name, g in groups.items():
    y = norm.pdf(x, g["mean"], g["sd"])
    neon = g["color"]
    BRIGHT = {"#4F7F84": "#8ED8DF", "#494C30": "#C8CCA0", "#F2CFD0": "#FFE0E1", "#C4787A": "#F5AAAC"}
    label_color = BRIGHT.get(neon, neon)
    r, gr, b = int(neon[1:3],16), int(neon[3:5],16), int(neon[5:7],16)

    # Filled area – line subtle, fill prominent
    fig.add_trace(go.Scatter(
        x=x, y=y,
        name=name,
        mode="lines",
        line=dict(color=f"rgba({r},{gr},{b},0.55)", width=1.5),
        fill="tozeroy",
        fillcolor=f"rgba({r},{gr},{b},0.55)",
        hovertemplate=(
            f"<b>{name}</b><br>"
            "IQ: %{x:.0f}<br>"
            "Density: %{y:.5f}"
            "<extra></extra>"
        ),
    ))

    # Mean vertical line
    peak_y = norm.pdf(g["mean"], g["mean"], g["sd"])
    fig.add_trace(go.Scatter(
        x=[g["mean"], g["mean"]],
        y=[0, peak_y],
        mode="lines",
        line=dict(color=f"rgba({r},{gr},{b},0.55)", width=1.5, dash="dot"),
        showlegend=False,
        hoverinfo="skip",
    ))

    # Mean label at peak
    fig.add_annotation(
        x=g["mean"], y=peak_y,
        text=f"\u03bc={g['mean']}",
        showarrow=False,
        font=dict(color=label_color, size=14, family="JetBrains Mono, Fira Code, monospace"),
        yshift=14,
        bgcolor="rgba(0,0,0,0.45)",
        borderpad=3,
    )

# ── Layout ─────────────────────────────────────────────────────────────
fig.update_layout(
    title=dict(
        text=(
            "<b>IQ Distribution in the United States by Race</b>"
            f'<br><span style="font-size:13px; color:{SUBTITLE_CLR}">'
            "Source: APA Task Force \u2014 <i>Intelligence: Knowns and Unknowns</i> (1996)  \u00b7  \u03c3 = 15 for all groups</span>"
        ),
        font=dict(color=TITLE_COLOR, size=22, family="Inter, Segoe UI, sans-serif"),
        x=0.5,
        xanchor="center",
    ),
    xaxis=dict(
        title=dict(text="IQ Score", font=dict(color=TEXT_COLOR, size=14)),
        tickfont=dict(color=TEXT_COLOR, size=12, family="JetBrains Mono, monospace"),
        range=[40, 160],
        dtick=10,
        gridcolor=GRID_COLOR,
        linecolor=AXIS_COLOR,
        zerolinecolor=AXIS_COLOR,
        showgrid=True,
        minor=dict(ticklen=3, tickcolor=AXIS_COLOR, dtick=5, showgrid=False),
    ),
    yaxis=dict(
        title=dict(text="Probability Density", font=dict(color=TEXT_COLOR, size=14)),
        tickfont=dict(color=TEXT_COLOR, size=12, family="JetBrains Mono, monospace"),
        gridcolor=GRID_COLOR,
        linecolor=AXIS_COLOR,
        zerolinecolor=AXIS_COLOR,
        showgrid=True,
        tickformat=".4f",
    ),
    plot_bgcolor=BG_COLOR,
    paper_bgcolor=PAPER_COLOR,
    font=dict(color=TEXT_COLOR),
    legend=dict(
        font=dict(color=TEXT_COLOR, size=13, family="Inter, sans-serif"),
        bgcolor="rgba(0,0,0,0.35)",
        bordercolor="rgba(255,235,200,0.15)",
        borderwidth=1,
        x=0.98, y=0.98,
        xanchor="right", yanchor="top",
    ),
    margin=dict(t=100, b=60, l=70, r=30),
    width=1100,
    height=620,
    hovermode="x unified",
)

# ── Neon glow effect on axes ──────────────────────────────────────────
# Add subtle reference lines at key IQ thresholds
for iq_val, label in [(70, "Borderline"), (85, "Low Avg"), (100, "Average"), (115, "High Avg"), (130, "Gifted")]:
    fig.add_vline(
        x=iq_val,
        line=dict(color="rgba(255,235,200,0.06)", width=1),
    )
    fig.add_annotation(
        x=iq_val, y=0,
        text=label,
        showarrow=False,
        font=dict(color="rgba(255,235,200,0.25)", size=9, family="Inter, sans-serif"),
        yshift=-22,
    )

# ── Export to output/ ─────────────────────────────────────────────────
os.makedirs('output', exist_ok=True)
fig.write_html('output/iq_distribution.html')
fig.write_image('output/iq_distribution.png', scale=2)
print('Saved \u2192 output/iq_distribution.html  &  output/iq_distribution.png')

fig.show()

Saved → output/iq_distribution.html  &  output/iq_distribution.png


In [26]:
# ══════════════════════════════════════════════════════════════════════
# Population-Weighted IQ Distribution
# Curves scaled so the area under each is proportional to actual
# U.S. population size for that racial group.
# Population: U.S. Census Bureau, ACS 2022 estimates
# ══════════════════════════════════════════════════════════════════════

# U.S. population by race (2022 ACS, in millions)
pop_millions = {
    "White":          196.0,   # White alone (non-Hispanic)
    "Hispanic":        63.6,   # Hispanic or Latino
    "Black":           41.1,   # Black or African American alone
    "Asian American":  20.0,   # Asian alone
}

# Combine IQ params + population
groups_pop = {
    "White":          {"mean": 100, "sd": 15, "color": "#494C30",  "pop": pop_millions["White"]},
    "Hispanic":       {"mean": 89,  "sd": 15, "color": "#F2CFD0",  "pop": pop_millions["Hispanic"]},
    "Black":          {"mean": 85,  "sd": 15, "color": "#C4787A",  "pop": pop_millions["Black"]},
    "Asian American": {"mean": 106, "sd": 15, "color": "#4F7F84",  "pop": pop_millions["Asian American"]},
}

x = np.linspace(40, 160, 600)

fig2 = go.Figure()

for name, g in groups_pop.items():
    # Scale: population (M) * pdf  ->  y = millions of people per IQ point
    y = g["pop"] * norm.pdf(x, g["mean"], g["sd"])
    neon = g["color"]
    BRIGHT = {"#4F7F84": "#8ED8DF", "#494C30": "#C8CCA0", "#F2CFD0": "#FFE0E1", "#C4787A": "#F5AAAC"}
    label_color = BRIGHT.get(neon, neon)
    r, gr, b = int(neon[1:3],16), int(neon[3:5],16), int(neon[5:7],16)

    # Filled curve – line subtle, fill prominent
    fig2.add_trace(go.Scatter(
        x=x, y=y,
        name=f"{name}  ({g['pop']:.1f}M)",
        mode="lines",
        line=dict(color=f"rgba({r},{gr},{b},0.25)", width=1.5),
        fill="tozeroy",
        fillcolor=f"rgba({r},{gr},{b},0.55)",
        hovertemplate=(
            f"<b>{name}</b>  (pop {g['pop']:.1f}M)<br>"
            "IQ: %{x:.0f}<br>"
            "\u2248 %{y:.2f}M people per IQ point"
            "<extra></extra>"
        ),
    ))

    # Mean dotted line
    peak_y = g["pop"] * norm.pdf(g["mean"], g["mean"], g["sd"])
    fig2.add_trace(go.Scatter(
        x=[g["mean"], g["mean"]], y=[0, peak_y],
        mode="lines",
        line=dict(color=f"rgba({r},{gr},{b},0.55)", width=1.5, dash="dot"),
        showlegend=False, hoverinfo="skip",
    ))

    # Peak annotation
    fig2.add_annotation(
        x=g["mean"], y=peak_y,
        text=f"\u03bc={g['mean']}  ({g['pop']:.0f}M)",
        showarrow=False,
        font=dict(color=label_color, size=13, family="JetBrains Mono, Fira Code, monospace"),
        yshift=14,
        bgcolor="rgba(0,0,0,0.50)",
        borderpad=3,
    )

# ── Layout ─────────────────────────────────────────────────────────────
fig2.update_layout(
    title=dict(
        text=(
            "<b>IQ Distribution \u2014 Weighted by Population Size</b>"
            f'<br><span style="font-size:13px; color:{SUBTITLE_CLR}">'
            "IQ: APA 1996  \u00b7  Population: ACS 2022  \u00b7  \u03c3 = 15  \u00b7  Curve area \u221d group population</span>"
        ),
        font=dict(color=TITLE_COLOR, size=22, family="Inter, Segoe UI, sans-serif"),
        x=0.5, xanchor="center",
    ),
    xaxis=dict(
        title=dict(text="IQ Score", font=dict(color=TEXT_COLOR, size=14)),
        tickfont=dict(color=TEXT_COLOR, size=12, family="JetBrains Mono, monospace"),
        range=[40, 160], dtick=10,
        gridcolor=GRID_COLOR, linecolor=AXIS_COLOR, zerolinecolor=AXIS_COLOR,
        showgrid=True,
        minor=dict(ticklen=3, tickcolor=AXIS_COLOR, dtick=5, showgrid=False),
    ),
    yaxis=dict(
        title=dict(text="People per IQ Point (millions)", font=dict(color=TEXT_COLOR, size=14)),
        tickfont=dict(color=TEXT_COLOR, size=12, family="JetBrains Mono, monospace"),
        gridcolor=GRID_COLOR, linecolor=AXIS_COLOR, zerolinecolor=AXIS_COLOR,
        showgrid=True,
    ),
    plot_bgcolor=BG_COLOR,
    paper_bgcolor=PAPER_COLOR,
    font=dict(color=TEXT_COLOR),
    legend=dict(
        font=dict(color=TEXT_COLOR, size=13, family="Inter, sans-serif"),
        bgcolor="rgba(0,0,0,0.35)",
        bordercolor="rgba(255,235,200,0.15)",
        borderwidth=1,
        x=0.98, y=0.98,
        xanchor="right", yanchor="top",
    ),
    margin=dict(t=100, b=60, l=70, r=30),
    width=1100,
    height=620,
    hovermode="x unified",
)

# IQ threshold labels
for iq_val, label in [(70, "Borderline"), (85, "Low Avg"), (100, "Average"), (115, "High Avg"), (130, "Gifted")]:
    fig2.add_vline(x=iq_val, line=dict(color="rgba(255,235,200,0.06)", width=1))
    fig2.add_annotation(
        x=iq_val, y=0, text=label, showarrow=False,
        font=dict(color="rgba(255,235,200,0.25)", size=9, family="Inter, sans-serif"),
        yshift=-22,
    )

# ── Export to output/ ─────────────────────────────────────────────────
fig2.write_html('output/iq_distribution_weighted.html')
fig2.write_image('output/iq_distribution_weighted.png', scale=2)
print('Saved \u2192 output/iq_distribution_weighted.html  &  output/iq_distribution_weighted.png')

fig2.show()

Saved → output/iq_distribution_weighted.html  &  output/iq_distribution_weighted.png
